# 4주차 ② 선형회귀 직접 구현 — 실습 5~6  ★ 과제 제출 대상

**목표**: `w -= lr * w.grad` 를 손으로 써서 선형회귀를 학습시키고,
같은 학습을 `nn.Linear` + `torch.optim.SGD` 로 다시 써서 **결과가 같음을 확인**한다.

> **실행 전 확인** — 우측 상단 커널 이름이 **`Python (dl2026)`** 인지 보세요.
> 아니면 커널명을 눌러 `Python (dl2026)` 을 선택하세요. (2주차 실습 8에서 다룬 그것입니다.)

---

## ★ 이 노트북에는 빈칸이 3곳 있습니다 (셀 2)

| 빈칸 | 채울 것 | 줄 수 |
|:---:|---|:---:|
| ① | 역전파 — `.grad` 를 채운다 | 1 |
| ② | 갱신 — 기울기 반대 방향으로 `lr` 만큼 | 2 |
| ③ | 지우기 — 누적된 `.grad` 를 0으로 | 2 |

> **빈칸을 안 채우고 실행하면** 손실이 **하나도 줄지 않고 같은 값이 반복**됩니다.
> 오류가 안 난다고 맞는 것이 아닙니다. **손실이 줄어드는지**를 보세요.

> **막히면**
> | 증상 | 원인 |
> |---|---|
> | 손실이 `13.4948` 로 계속 같다 | 빈칸을 하나도 안 채웠다 |
> | `AttributeError: 'NoneType' object has no attribute 'zero_'` | ①을 안 채웠다 (`.grad` 가 아직 `None`) |
> | 손실이 **줄었다 늘었다를 반복**한다 | ③을 안 채웠다 (기울기 누적 — 1교시 실습 3의 그 증상) |
> | 손실이 `nan` | `lr` 을 바꿨다면 되돌리세요 (0.1) |
> | 손실이 거의 안 줄어든다 | ②가 `with torch.no_grad():` 블록 **밖**에 있다 |

> **정답을 채우면** `w = 3.0024`, `b = 2.0071` 이 나옵니다 (시드 42 기준).

## 실습 5 — 선형회귀 직접 구현

In [ ]:
# 셀 1 — 데이터 만들기 (정답을 우리가 정한다)
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

torch.manual_seed(42)
N = 100
x = torch.randn(N, 1)                          # ★ 평균 0 근처의 값
y = 3 * x + 2 + 0.2 * torch.randn(N, 1)        # 정답: w = 3, b = 2

print("x.shape :", x.shape, "| y.shape :", y.shape)
plt.scatter(x, y, s=12)
plt.xlabel("x"); plt.ylabel("y"); plt.title("만든 데이터 (정답: y = 3x + 2)")
plt.show()

In [ ]:
# 셀 2 — 학습 루프  ★★ 빈칸 3곳을 채우세요
w = torch.zeros(1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)
lr = 0.1
history = []

for epoch in range(200):
    # ① 순전파 — 예측한다
    pred = x * w + b

    # ② 손실 — 얼마나 틀렸나
    loss = ((pred - y) ** 2).mean()

    # ─────────────────────────────────────────────────────────
    # 빈칸 ①  역전파 — .grad 를 채운다   (한 줄)
    #         힌트:  loss.______()
    # ─────────────────────────────────────────────────────────


    with torch.no_grad():
        # ─────────────────────────────────────────────────────
        # 빈칸 ②  갱신 — 기울기 반대 방향으로 lr 만큼   (두 줄)
        #         힌트:  w -= lr * w.____     b 도 같이
        # ─────────────────────────────────────────────────────
        pass

        # ─────────────────────────────────────────────────────
        # 빈칸 ③  지우기 — 누적을 없앤다   (두 줄)
        #         힌트:  w.grad.______()      b 도 같이
        # ─────────────────────────────────────────────────────
        pass

    history.append(loss.item())
    if epoch % 20 == 0:
        print(f"epoch {epoch:3d} | loss {loss.item():8.4f} | w {w.item():6.3f} | b {b.item():6.3f}")

print(f"\n학습 완료 → w = {w.item():.4f}, b = {b.item():.4f}   (정답: 3, 2)")

In [ ]:
# 셀 3 — 학습이 잘 됐는지 두 그림으로 본다
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

ax[0].plot(history)
ax[0].set_title("손실 곡선"); ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss")

ax[1].scatter(x, y, s=12, label="데이터")
with torch.no_grad():                                   # ★ 평가에는 그래프가 필요 없다
    xs = torch.linspace(x.min(), x.max(), 50).reshape(-1, 1)
    ax[1].plot(xs, xs * w + b, color="red",
               label=f"학습된 직선 (w={w.item():.2f}, b={b.item():.2f})")
ax[1].legend(); ax[1].set_title("학습 결과")

plt.tight_layout(); plt.show()

왼쪽 손실 곡선이 **부드럽게 내려가 평평해지고**, 오른쪽 빨간 직선이 점들의 한가운데를 지나가면 성공입니다.

> 셀 3에서 `with torch.no_grad():` 를 또 썼습니다. 이번엔 갱신이 아니라 **평가·시각화** 때문입니다.
> 학습이 아닌 계산에는 그래프가 필요 없고, 감싸 주면 메모리도 아낍니다.

## 실습 6 — `torch.optim` 으로 리팩터링

In [ ]:
# 셀 4 — 같은 학습을 nn.Linear + SGD 로 (실습 6)
torch.manual_seed(42)                       # 초기값을 고정해 비교 가능하게
model = nn.Linear(1, 1)
loss_fn = nn.MSELoss()                      # ((pred - y)**2).mean() 과 같다
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

history_optim = []

for epoch in range(200):
    pred = model(x)                         # ① 순전파
    loss = loss_fn(pred, y)                 # ② 손실

    optimizer.zero_grad()                   # ⑤ 지우기 (앞으로 옮겼다)
    loss.backward()                         # ③ 역전파
    optimizer.step()                        # ④ 갱신  ← 두 줄이 한 줄로

    history_optim.append(loss.item())
    if epoch % 20 == 0:
        print(f"epoch {epoch:3d} | loss {loss.item():8.4f} "
              f"| w {model.weight.item():6.3f} | b {model.bias.item():6.3f}")

print(f"\noptim 결과 → w = {model.weight.item():.4f}, b = {model.bias.item():.4f}   (정답: 3, 2)")

**학습 루프의 다섯 단계가 그대로** 있습니다. 이름만 바뀌었습니다.

```
loss.backward()          →  그대로
w -= lr * w.grad  (×2)   →  optimizer.step()
w.grad.zero_()    (×2)   →  optimizer.zero_grad()
with torch.no_grad():    →  사라짐 (step() 안에서 처리)
```

In [ ]:
# 셀 5 — 직접 구현 vs optim  ★ 과제 핵심
print(f"직접 구현 : w = {w.item():.4f}, b = {b.item():.4f}")
print(f"optim     : w = {model.weight.item():.4f}, b = {model.bias.item():.4f}")

plt.figure(figsize=(6, 4))
plt.plot(history,       label="직접 구현 (2교시)")
plt.plot(history_optim, label="torch.optim", linestyle="--")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend()
plt.title("두 방식의 손실 곡선")
plt.show()

**최종 값이 소수점 넷째 자리까지 똑같이 나옵니다.**

```
직접 구현 : w = 3.0024, b = 2.0071
optim     : w = 3.0024, b = 2.0071
```

출발점은 달랐습니다 — 직접 구현은 `w=0, b=0` 에서, `nn.Linear` 는 **무작위 초기값**에서 시작했습니다.
실제로 **초반 손실 곡선은 서로 다릅니다** (epoch 0 에서 13.49 vs 6.57).
그런데도 40 epoch 쯤부터 두 곡선이 포개지고, 최종 값이 같습니다.

> **왜 같은가 ★**: 이 손실 곡면은 **바닥이 하나뿐인 그릇 모양**입니다.
> 어디서 굴러 내려가든 **같은 바닥**에 도착합니다.
> (5주차부터 층을 쌓으면 이 성질이 깨집니다 — 그때는 출발점이 결과를 바꿉니다.)

> `optimizer.step()` 은 블랙박스가 아닙니다. 셀 2에서 여러분이 손으로 쓴 그 두 줄입니다.

---

### 과제 제출 전 확인

- [ ] 셀 1~5가 모두 실행되고 **출력이 저장된 상태**다 (`Ctrl+S`)
- [ ] 셀 2 결과가 `w ≈ 3`, `b ≈ 2` 다
- [ ] 셀 5에서 두 방식의 결과를 **나란히 비교**했다 ★ 과제 핵심
- [ ] 회고 3줄 — 특히 *"`zero_grad()` 를 빼면 왜 망가지는가"* 를 자기 말로